In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from pathlib import Path
import time
import json
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')


Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU Memory: 6.00 GB


In [2]:
# Dataset paths
data_dir = Path('datasets_paper/2_class_dataset')
train_dir = data_dir

# Class names
class_names = ['Anomaly', 'No-Anomaly']
num_classes = len(class_names)

print(f"Dataset directory: {data_dir}")
print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")

# Count images in each class
for class_name in class_names:
    class_path = data_dir / class_name
    if class_path.exists():
        num_images = len(list(class_path.glob('*.jpg')))
        print(f"{class_name}: {num_images} images")
    else:
        print(f"Warning: {class_path} does not exist")


Dataset directory: datasets_paper\2_class_dataset
Number of classes: 2
Classes: ['Anomaly', 'No-Anomaly']
Anomaly: 20000 images
No-Anomaly: 20000 images


In [3]:
# Hyperparameters matching ViT notebook configuration
HYPERPARAMS = {
    'input_shape': (160, 160, 3),
    'num_classes': 2,
    'batch_size': 8,
    'epochs': 60,
    'learning_rate': 1e-4,
    'learning_rate_decay': 0.2,
    'patience_lr': 5,
    'patience_early_stop': 9,
    'validation_split': 0.1,
    'test_split': 0.1,
    'label_smoothing': 0.2,
    'epsilon': 1e-6,
    'weight_decay': 0.01
}

# Extract commonly used variables
image_size = HYPERPARAMS['input_shape'][0]  # 160
batch_size = HYPERPARAMS['batch_size']  # 8

# Data transforms (matching ViT configuration exactly)
train_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMS['input_shape'][0], HYPERPARAMS['input_shape'][1])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMS['input_shape'][0], HYPERPARAMS['input_shape'][1])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"Image size: {image_size}x{image_size}")
print(f"Batch size: {batch_size}")
print(f"\nHyperparameters:")
for key, value in HYPERPARAMS.items():
    print(f"  {key}: {value}")


Image size: 160x160
Batch size: 8

Hyperparameters:
  input_shape: (160, 160, 3)
  num_classes: 2
  batch_size: 8
  epochs: 60
  learning_rate: 0.0001
  learning_rate_decay: 0.2
  patience_lr: 5
  patience_early_stop: 9
  validation_split: 0.1
  test_split: 0.1
  label_smoothing: 0.2
  epsilon: 1e-06
  weight_decay: 0.01


In [4]:
# DeiT Classifier matching ViT architecture
class DeiTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True, dropout_rate=0.3):
        super(DeiTClassifier, self).__init__()
        
        # Load DeiT-B/16 model (closest to ViT-B/32 architecture)
        self.backbone = timm.create_model(
            'deit_base_patch16_224',  # DeiT-B/16 
            pretrained=pretrained,
            num_classes=0,  # Remove head, we'll add custom one
            img_size=160    # Match ViT input size
        )
        
        # Get the feature dimension (768 for DeiT-B)
        self.feature_dim = self.backbone.num_features
        
        # Custom classification head (matching ViT architecture exactly)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, 0, 0.01)
                torch.nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Extract features using DeiT backbone
        features = self.backbone(x)
        # Classification
        output = self.classifier(features)
        return output

# Selected model configuration
selected_model = 'deit_base_patch16_224'  # DeiT-B/16 (closest equivalent to ViT-B/32)
print(f"Selected model: {selected_model}")
print(f"Note: Using DeiT-B/16 with 160x160 input (closest to ViT-B/32 configuration)")

# Create model
model = DeiTClassifier(num_classes=HYPERPARAMS['num_classes'], pretrained=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Summary:")
print(f"Architecture: DeiT-B/16 with custom head")
print(f"Input size: {HYPERPARAMS['input_shape']}")
print(f"Feature dimension: {model.feature_dim}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024**2:.2f} MB (fp32)")


Selected model: deit_base_patch16_224
Note: Using DeiT-B/16 with 160x160 input (closest to ViT-B/32 configuration)

Model Summary:
Architecture: DeiT-B/16 with custom head
Input size: (160, 160, 3)
Feature dimension: 768
Total parameters: 86,119,682
Trainable parameters: 86,119,682
Model size: 328.52 MB (fp32)


In [5]:
# Load full dataset
full_dataset = ImageFolder(root=data_dir, transform=train_transform)

# Calculate split sizes (80% train, 10% validation, 10% test - matching ViT)
total_size = len(full_dataset)
test_size = int(HYPERPARAMS['test_split'] * total_size)
val_size = int(HYPERPARAMS['validation_split'] * total_size)
train_size = total_size - test_size - val_size

print(f"Total dataset size: {total_size}")
print(f"Train size: {train_size} ({train_size/total_size*100:.1f}%)")
print(f"Validation size: {val_size} ({val_size/total_size*100:.1f}%)")
print(f"Test size: {test_size} ({test_size/total_size*100:.1f}%)")

# Split dataset (matching ViT configuration)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply different transforms to validation and test sets
val_dataset.dataset = ImageFolder(root=data_dir, transform=val_transform)
test_dataset.dataset = ImageFolder(root=data_dir, transform=val_transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=HYPERPARAMS['batch_size'], 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=HYPERPARAMS['batch_size'], 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=HYPERPARAMS['batch_size'], 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

print(f"\nData loaders created:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Total dataset size: 40000
Train size: 32000 (80.0%)
Validation size: 4000 (10.0%)
Test size: 4000 (10.0%)

Data loaders created:
Training batches: 4000
Validation batches: 500
Test batches: 500


In [6]:
# Label Smoothing Cross Entropy (matching ViT implementation)
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing
        
    def forward(self, pred, target):
        confidence = 1. - self.smoothing
        log_probs = torch.log_softmax(pred, dim=1)
        nll_loss = -log_probs.gather(dim=1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -log_probs.mean(dim=1)
        loss = confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

# Training configuration (matching ViT hyperparameters exactly)
num_epochs = HYPERPARAMS['epochs']
learning_rate = HYPERPARAMS['learning_rate']
weight_decay = HYPERPARAMS['weight_decay']

# Loss function with label smoothing
criterion = LabelSmoothingCrossEntropy(smoothing=HYPERPARAMS['label_smoothing'])

# Optimizer (AdamW matching ViT configuration)
optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS['learning_rate'],
    weight_decay=HYPERPARAMS['weight_decay'],
    eps=HYPERPARAMS['epsilon']
)

# Learning rate scheduler (ReduceLROnPlateau matching ViT)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=HYPERPARAMS['learning_rate_decay'],
    patience=HYPERPARAMS['patience_lr'],
    verbose=True,
    min_lr=1e-7
)

print(f"Training Configuration (matching ViT setup):")
print(f"Number of epochs: {num_epochs}")
print(f"Learning rate: {learning_rate}")
print(f"Weight decay: {weight_decay}")
print(f"Label smoothing: {HYPERPARAMS['label_smoothing']}")
print(f"Optimizer: AdamW")
print(f"Scheduler: ReduceLROnPlateau")
print(f"Early stopping patience: {HYPERPARAMS['patience_early_stop']}")


Training Configuration (matching ViT setup):
Number of epochs: 60
Learning rate: 0.0001
Weight decay: 0.01
Label smoothing: 0.2
Optimizer: AdamW
Scheduler: ReduceLROnPlateau
Early stopping patience: 9


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
best_model_path = 'best_deit_2class_model.pth'

print("Starting training...\\n")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'model_name': selected_model
        }, best_model_path)
        print(f"✅ New best model saved with validation accuracy: {val_acc:.2f}%")
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"Learning Rate: {current_lr:.6f}")
    print()

total_time = time.time() - start_time
print(f"Training completed in {total_time//3600:.0f}h {(total_time%3600)//60:.0f}m {total_time%60:.0f}s")
print(f"Best validation accuracy: {best_val_acc:.2f}%")


Starting training...\n
Epoch [1/60]
--------------------------------------------------


Validation: 100%|██████████| 500/500 [00:33<00:00, 14.79it/s, Loss=0.4438, Acc=85.20%] 


✅ New best model saved with validation accuracy: 85.20%
Train Loss: 0.5361, Train Acc: 80.79%
Val Loss: 0.4948, Val Acc: 85.20%
Learning Rate: 0.000100

Epoch [2/60]
--------------------------------------------------


Validation: 100%|██████████| 500/500 [00:36<00:00, 13.58it/s, Loss=0.4491, Acc=88.17%] 


✅ New best model saved with validation accuracy: 88.17%
Train Loss: 0.4871, Train Acc: 86.19%
Val Loss: 0.4568, Val Acc: 88.17%
Learning Rate: 0.000100

Epoch [3/60]
--------------------------------------------------


...

Validation: 100%|██████████| 500/500 [00:36<00:00, 13.52it/s, Loss=0.3874, Acc=93.08%] 


Train Loss: 0.4037, Train Acc: 93.64%
Val Loss: 0.4067, Val Acc: 93.08%
Learning Rate: 0.000100

Epoch [13/60]
--------------------------------------------------


Validation: 100%|██████████| 500/500 [00:35<00:00, 13.91it/s, Loss=0.3471, Acc=93.67%] 


✅ New best model saved with validation accuracy: 93.67%
Train Loss: 0.4010, Train Acc: 93.84%
Val Loss: 0.4046, Val Acc: 93.67%
Learning Rate: 0.000100

Epoch [14/60]
--------------------------------------------------


Training:  32%|███▏      | 1293/4000 [03:56<06:17,  7.18it/s, Loss=0.5045, Acc=94.62%]

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('DeiT 2-Class Training History', fontsize=16)

epochs_range = range(1, num_epochs + 1)

# Plot loss
axes[0, 0].plot(epochs_range, history['train_loss'], label='Training Loss', color='blue')
axes[0, 0].plot(epochs_range, history['val_loss'], label='Validation Loss', color='red')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Plot accuracy
axes[0, 1].plot(epochs_range, history['train_acc'], label='Training Accuracy', color='blue')
axes[0, 1].plot(epochs_range, history['val_acc'], label='Validation Accuracy', color='red')
axes[0, 1].set_title('Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Plot learning rate
axes[1, 0].plot(epochs_range, history['lr'], color='green')
axes[1, 0].set_title('Learning Rate')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True)

# Plot accuracy difference
acc_diff = [train - val for train, val in zip(history['train_acc'], history['val_acc'])]
axes[1, 1].plot(epochs_range, acc_diff, color='purple')
axes[1, 1].set_title('Overfitting Monitor (Train - Val Accuracy)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy Difference (%)')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Load best model for evaluation
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} with validation accuracy: {checkpoint['val_acc']:.2f}%")

# Evaluate on validation set
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Final Evaluation'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate final metrics
accuracy = accuracy_score(all_labels, all_predictions)
print(f"\\nFinal Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Classification report
print("\\nClassification Report:")
print(classification_report(all_labels, all_predictions, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - DeiT 2-Class Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Save results
results = {
    'model_name': selected_model,
    'final_accuracy': float(accuracy),
    'best_val_accuracy': float(best_val_acc),
    'training_time_minutes': total_time / 60,
    'total_parameters': total_params
}

with open('deit_2class_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f"\\n✅ Training completed successfully!")
print(f"📊 Final Summary:")
print(f"Model: {selected_model}")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Final test accuracy: {accuracy*100:.2f}%")
print(f"Training time: {total_time//60:.0f} minutes")
print(f"Model parameters: {total_params:,}")


In [ ]:
# Create and plot confusion matrix
# Create confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - DeiT 2-Class Classification', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)

# Add percentage annotations
total = cm.sum()
for i in range(len(class_names)):
    for j in range(len(class_names)):
        percentage = (cm[i, j] / total) * 100
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

# Print confusion matrix statistics
print("\nConfusion Matrix Analysis:")
print(f"True Negatives (TN): {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP): {cm[1,1]}")

# Calculate additional metrics
if len(class_names) == 2:
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\nDetailed Metrics:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1-Score: {f1:.4f}")


In [ ]:
# Also generate ROC/AUC Curve
# Generate ROC/AUC Curves
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get prediction probabilities for ROC curve
model.eval()
all_probs = []
all_labels_roc = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        
        all_probs.extend(probabilities.cpu().numpy())
        all_labels_roc.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_labels_roc = np.array(all_labels_roc)

# Calculate ROC curve and AUC for each class
plt.figure(figsize=(12, 5))

# For binary classification
if len(class_names) == 2:
    # ROC curve for positive class (class 1)
    fpr, tpr, _ = roc_curve(all_labels_roc, all_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - DeiT 2-Class Classification')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    # Precision-Recall curve
    from sklearn.metrics import precision_recall_curve, average_precision_score
    
    precision_vals, recall_vals, _ = precision_recall_curve(all_labels_roc, all_probs[:, 1])
    avg_precision = average_precision_score(all_labels_roc, all_probs[:, 1])
    
    plt.subplot(1, 2, 2)
    plt.plot(recall_vals, precision_vals, color='darkgreen', lw=2,
             label=f'PR curve (AP = {avg_precision:.3f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nROC AUC Score: {roc_auc:.4f}")
    print(f"Average Precision Score: {avg_precision:.4f}")

else:
    # Multi-class ROC curves
    plt.figure(figsize=(10, 8))
    
    # Binarize labels for multi-class ROC
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(all_labels_roc, classes=range(len(class_names)))
    
    # Calculate ROC curve for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
        plt.plot(fpr[i], tpr[i], lw=2,
                label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multi-class ROC Curves - DeiT Classification')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Print AUC scores for each class
    print("\nAUC Scores by Class:")
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {roc_auc[i]:.4f}")
    
    # Calculate macro and micro average AUC
    macro_auc = np.mean(list(roc_auc.values()))
    print(f"\nMacro Average AUC: {macro_auc:.4f}")


In [ ]:
# Also generate the t-SNE plot for the 2-class classification
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Extract features from the model for t-SNE visualization
model.eval()
features_list = []
labels_list = []

print("Extracting features for t-SNE visualization...")
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Extracting features"):
        images, labels = batch
        images = images.to(device)
        
        # Get features from the model (before the final classification layer)
        # For DeiT, we'll use the CLS token representation
        outputs = model(images)
        # Extract the hidden states (features before classification)
        features = model.deit.embeddings(images)
        features = model.deit.encoder(features)
        # Use the CLS token (first token)
        cls_features = features.last_hidden_state[:, 0, :]  # Shape: [batch_size, hidden_size]
        
        features_list.append(cls_features.cpu().numpy())
        labels_list.append(labels.cpu().numpy())

# Concatenate all features and labels
all_features = np.concatenate(features_list, axis=0)
all_labels_tsne = np.concatenate(labels_list, axis=0)

# Apply t-SNE
print("Applying t-SNE dimensionality reduction...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(all_features)

# Create the t-SNE plot
plt.figure(figsize=(10, 8))
colors = ['red', 'blue']  # Colors for binary classification
for i, class_name in enumerate(class_names):
    mask = all_labels_tsne == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1], 
                c=colors[i], label=class_name, alpha=0.6, s=20)

plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Visualization of DeiT Features - Binary Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"t-SNE visualization completed. Plotted {len(all_features)} samples.")
